In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import glob
import os
from pathlib import Path
import sys
import pandas as pd
import corner

import cogwheel.likelihood
import cogwheel.data
import cogwheel.sampling
import gwosc
import gwpy.table
from pesummary.io import read

from cogwheel.likelihood.marginalization.coherent_score_lensing import CoherentScoreLensing

sys.path.insert(0, '../code')
import make_event_data
from parameter_estimation import get_chirp_mass, get_table_data, add_derived_quantities, check_eventname, get_event_data

### get event data

In [3]:
# which catalog
catalog = 'GWTC-4.0'
# where to look for event data
catalog_dir = f'/home/javier.roulet/cogwheel-catalog/processed_data/{catalog}'
# pick an event from the catalog
eventname = 'GW230628_231200'

In [4]:
# get event data in the table
event_table = get_table_data(eventname, catalog)
eventname = check_eventname(eventname, event_table)

In [5]:
# event data config
config_fn = os.path.join(catalog_dir, eventname, 'event_data_kwargs.json')
with open(config_fn) as file:
    event_pars = json.load(file)

t_merger_guess = 0.
print(f"t_merger_guess = {t_merger_guess:.2f} sec", flush=True)
mchirp_guess = get_chirp_mass(event_table)
print(f"detector-frame chirp mass guess = {mchirp_guess:.2f} Msun", flush=True)

t_merger_guess = 0.00 sec
detector-frame chirp mass guess = 35.70 Msun


In [6]:
# get the event data
event_data = get_event_data(eventname, **event_pars)

Skipping existing file /home/abbye.williams/cogwheel/cogwheel/data/gwosc_files/GW230628_231200/H_GW230628_231200.hdf5
Skipping existing file /home/abbye.williams/cogwheel/cogwheel/data/gwosc_files/GW230628_231200/L_GW230628_231200.hdf5


### posterior

In [7]:
# approximant
approximant = 'IMRPhenomXPHM'
# prior class
prior_class = 'IntrinsicLVCPrior'
# run directory
rundir = '../data/pe_runs_lensing'

In [9]:
# first we want to perform a fast likelihood maximization to find our reference waveform:
posterior = cogwheel.posterior.Posterior.from_event(
    event_data,
    mchirp_guess,
    approximant,
    prior_class,
    ref_wf_finder_kwargs={
        'f_ref': 100.0,  # Just so it matches the injection and it makes sense to compare parameters
        'time_range': (t_merger_guess - 0.1, t_merger_guess + 0.1)  # Edit if needed
    }
)

Searching incoherent solution for GW230628_231200
Set intrinsic parameters, lnL = 122.17171743660643
Set time, lnL(H) = 77.21384326328214
Set sky location, lnL = 122.28282730071747
Set phase and distance, lnL = 122.28282730071747
Set mchirp_range = (np.float64(20.127996485177018), np.float64(123.72682075034415))
likelihood.coherent_score.__class__:  <class 'cogwheel.likelihood.marginalization.coherent_score_hm.CoherentScoreHM'>


In [10]:
# instantiate a CoherentScoreLensing object
coherent_score = CoherentScoreLensing(**posterior.likelihood.coherent_score.get_init_dict())
# reinstantiate the likelihood
lensing_likelihood = posterior.likelihood.reinstantiate(coherent_score=coherent_score)

In [11]:
# now we have to reinstantiate the posterior
# !! there must be a better way to do this... maybe we don't need the init_dict explicitly?
#   can set manually, use default args, etc.
posterior = cogwheel.posterior.Posterior.from_event(
    event_data,
    mchirp_guess,
    approximant,
    prior_class,
    likelihood_class=lensing_likelihood,
    ref_wf_finder_kwargs={
        'f_ref': 100.0,  # Just so it matches the injection and it makes sense to compare parameters
        'time_range': (t_merger_guess - 0.1, t_merger_guess + 0.1)  # Edit if needed
    }
)

Searching incoherent solution for GW230628_231200
Set intrinsic parameters, lnL = 122.17171743660643
Set time, lnL(H) = 77.21384326328214
Set sky location, lnL = 122.28282730071747
Set phase and distance, lnL = 122.28282730071747
Set mchirp_range = (np.float64(20.127996485177018), np.float64(123.72682075034415))
likelihood.coherent_score.__class__:  <class 'cogwheel.likelihood.marginalization.coherent_score_lensing.CoherentScoreLensing'>


In [13]:
posterior.likelihood.coherent_score.__class__

cogwheel.likelihood.marginalization.coherent_score_lensing.CoherentScoreLensing

### sampler

In [14]:
# using Nautilus
sampler = cogwheel.sampling.Nautilus(posterior)

In [15]:
sampler.posterior.likelihood.coherent_score.__class__

cogwheel.likelihood.marginalization.coherent_score_lensing.CoherentScoreLensing

In [17]:
# get the run directory
rundir = sampler.get_rundir(parentdir=rundir)

In [18]:
sampler._get_sampler_kwargs()

{'prior': <nautilus.prior.Prior at 0x7f80efac5410>,
 'likelihood': <bound method Sampler._lnfoldedprob_and_blob of <cogwheel.sampling.Nautilus object at 0x7f80ed9a6c10>>,
 'blobs_dtype': [('f_ref', float),
  ('m1', numpy.float64),
  ('m2', numpy.float64),
  ('s1z', numpy.float64),
  ('s2z', numpy.float64),
  ('iota', float),
  ('s1x_n', float),
  ('s1y_n', float),
  ('s2x_n', float),
  ('s2y_n', float),
  ('l1', float),
  ('l2', float),
  ('d_luminosity', numpy.float64),
  ('dec', numpy.float64),
  ('lon', numpy.float64),
  ('phi_ref', numpy.float64),
  ('psi', numpy.float64),
  ('t_geocenter', numpy.float64),
  ('lnl_marginalized', numpy.float64),
  ('lnl', numpy.float64),
  ('h_h', numpy.float64),
  ('n_effective', numpy.float64),
  ('n_qmc', numpy.int64),
  ('p_lensed', numpy.float64),
  ('ra', numpy.float64)],
 'pass_dict': False}

In [19]:
# run the sampling
sampler.run(rundir)

Starting the nautilus sampler...
Please report issues at github.com/johannesulf/nautilus.
Status    | Bounds | Ellipses | Networks | Calls    | f_live | N_eff | log Z    
Finished  | 19     | 3        | 4        | 52900    | N/A    | 19329 | +86.30   


In [21]:
# get the samples
samples = pd.read_feather(rundir/'samples.feather')